In [4]:
!pip install -U yfinance -q

In [5]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import date
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')


def data(Ticker  , start , end) :
  x = yf.download(Ticker  , start , end , multi_level_index=False)
  x = x.ffill()
  return x

df1 = data('HDFCBANK.NS' , '2020-01-01' , date.today())
df2 = data('TCS.NS' , '2020-01-01' , date.today())
df3 = data('RELIANCE.NS' , '2020-01-01' , date.today())
df4 = data('SUNPHARMA.NS' , '2020-01-01' , date.today())
df5 = data('HINDUNILVR.NS' , '2020-01-01' , date.today())
df6 = data('BHARTIARTL.NS' , '2020-01-01' , date.today())
df7 = data('M&M.NS' , '2020-01-01' , date.today())
df8 = data('LT.NS' , '2020-01-01' , date.today())
df_b = data('^NSEI' , '2020-01-01' , date.today())

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [6]:
#backtest

def backtest(signal , close, fee=0.0025 , alloc = 1.0 , capital = 1000000) :
  shares = ( (capital * alloc) / close.shift(1) ).fillna(0).astype(int)
  daily_pnl = close.diff() * signal.shift(1) * shares
  position = signal.diff().fillna(0)
  daily_fees = fee * shares * position.shift(1).abs() * close.shift(1)
  equity = capital + (daily_pnl - daily_fees).cumsum()
  return equity

In [7]:

#Moving Average

capital = 1000000
def Moving_Average(df , wind_s1 , wind_s2) :
  SMA_fast = df['Close'].rolling(window = wind_s1).mean()
  SMA_slow = df['Close'].rolling(window = wind_s2).mean()

  Signal = (SMA_fast > SMA_slow).astype(int)

  return Signal

#Breakout Strategy

def Breakout_strategy(df , window_sz) :
  wind_high = df['Close'].shift(1).rolling(window = window_sz).max()
  wind_low = df['Close'].shift(1).rolling(window = window_sz).min()

  raw_signal = np.select(
      [df['Close'] > wind_high , df['Close'] < wind_low] ,
      [1,0] ,
      default = np.nan
  )

  signal = pd.Series(raw_signal, index=df.index).ffill().fillna(0)

  return signal

#RSI mean reversion Strategy

def RSI_MR(df , wind_sz) :

  delta = df['Close'].diff()

  gain = delta.clip(lower = 0)
  loss = -delta.clip(upper = 0)

  avg_gain = gain.rolling(window = wind_sz).mean()
  avg_loss = loss.rolling(window = wind_sz).mean()

  rs = avg_gain / avg_loss

  rs_val = 100 - (100 / (1 +  rs))

  raw_signal = np.select(
      [(rs_val.shift(1) < 30) & (rs_val >= 30) , (rs_val.shift(1) < 50) & (rs_val >= 50)] ,
      [1 , 0] ,
      default = np.nan
  )

  signal = pd.Series(raw_signal , index = df.index).ffill().fillna(0)

  return signal

In [8]:
stocks = {
    "HDFCBANK": df1,
    "TCS": df2,
    "RELIANCE": df3,
    "SUNPHARMA": df4,
    "HINDUNILVR": df5,
    "BHARTIARTL": df6,
    "M&M": df7,
    "LT": df8,
}


In [9]:
def performance_report(signal, close) :
  # print(close)
  fee = 0.0025

  strategy_ret = signal.shift(1) * close.pct_change() - fee * signal.diff().shift(1).abs().fillna(0)
  cumm_strategy_ret = (1 + strategy_ret).cumprod()


  N = signal.shape[0]
  rf = 0.065
  daily_rf = rf/252

  ann_ret = (cumm_strategy_ret.iloc[-1]) ** (252 / N) - 1
  ann_vol = strategy_ret.std() * np.sqrt(252)
  sharpe =  (ann_ret - rf) / (ann_vol)

  downside_diff = np.minimum(0, strategy_ret - daily_rf)
  downside_dev = np.sqrt((downside_diff ** 2).mean()) * np.sqrt(252)

  sortino = (ann_ret - rf) / downside_dev

  #drawdown analysis:
  dd = (cumm_strategy_ret - cumm_strategy_ret.cummax()) / cumm_strategy_ret.cummax()
  max_dd = dd.min()
  max_dd_date = dd.idxmin()

  avg_dd = dd.mean()

  running_max = cumm_strategy_ret.cummax()

  trough_date = dd.idxmin()
  peak_value = running_max.loc[trough_date]

  # 2. Slice strictly from the trough forward and filter for full recovery
  post_trough = cumm_strategy_ret.loc[trough_date:]
  recovered = post_trough[post_trough >= peak_value]

  # print(recovered)

  if recovered.empty:
    recovery_days = np.nan
  else:
    recovery_date = recovered.index[0]
    recovery_days = len(post_trough.loc[:recovered.index[0]]) - 1

  pos = signal.shift(1).fillna(0)
  days_invested = pos.cumsum().iloc[-1]
  exposure = 1 - pos.mean()   #fraction of time invested
  cash_pct = 1 - exposure     #fraction of time in cash

  return {
      "Cumulative Return": cumm_strategy_ret.iloc[-1] - 1,
      "Annual Return": ann_ret,
      "Annual Volatility": ann_vol,
      "Sharpe": sharpe,
      "Sortino": sortino,
      "Max Drawdown": max_dd,
      "Max DD Date": max_dd_date,
      "Avg Drawdown": avg_dd,
      "Recovery Days": recovery_days,
  }

# performance_report(signals['RELIANCE']['MA'], df3['Close'])

In [10]:
def trade_table(signal, close, fee=0.0025) :
  pos = signal.diff().fillna(signal.iloc[0])

  entry_date = pd.Series(close.index , index = close.index).where(pos == 1).ffill()
  entry_price = close.where(pos==1).ffill()

  df = pd.DataFrame( {
      'Entry_date' : entry_date ,
      'Entry_price' : entry_price ,
      'Exit_price' : close ,
      'Position' : pos ,
  } , index = close.index)

  df = df[['Entry_date' , 'Entry_price' , 'Exit_price' , 'Position']][df['Position'] == -1]

  Num_of_trades = df.shape[0]

  if Num_of_trades == 0:
        return {
            "Total Trades": 0,
            "Win Rate": np.nan,
            "Avg Win %": np.nan,
            "Avg Loss %": np.nan,
            "Largest Win %": np.nan,
            "Largest Loss %": np.nan,
            "Avg Holding Days": np.nan,
        }

  df['Profit'] =  (df['Exit_price'] * (1 - fee)) /  (df['Entry_price'] * (1 + fee)) - 1
  df['Holding_period'] = (df.index - df['Entry_date'])

  win = df['Profit'].clip(lower = 0)

  Win_trades = (df['Profit'] > 0).sum()
  Win_rate = 0 if Num_of_trades == 0 else Win_trades / Num_of_trades

  Total_win = win.cumsum().iloc[-1]
  # avg_win = Total_win / (win > 0).sum()

  loss = -df['Profit'].clip(upper = 0)
  # Total_loss = loss.cumsum().iloc[-1]
  # avg_loss = Total_loss / (loss > 0).sum()

  # Largest_win = win.cummax().iloc[-1]
  # Largest_loss = loss.cummax().iloc[-1]

  Total_days = (close.index[-1] - close.index[0])
  # Avg_holding_period = (df['Holding_period'].cumsum().iloc[-1]) / Num_of_trades
  Avg_holding_period = df['Holding_period'].dt.days.mean()

  return {
      "Total Trades": Num_of_trades,
      "Win Rate": Win_rate,
      "Avg Win %": win.mean(),
      "Avg Loss %": loss.mean(),
      "Largest Win %": win.max(),
      "Largest Loss %": loss.max(),
      "Avg Holding Days": Avg_holding_period,
  }

# trade_table(signals['RELIANCE']['MA'], df3['Close'])

In [11]:
signals = {}
results = {}
summary_rows = {}

for stock, df in stocks.items():
    close = df['Close']

    sigs = {
        "MA": Moving_Average(df, 20, 50),
        "Breakout": Breakout_strategy(df, 20),
        "RSI": RSI_MR(df, 14),
        "Buy & Hold": pd.Series(1, index=df.index),
    }
    signals[stock] = sigs

    # Equity curves from backtest()
    results[stock] = pd.DataFrame(
        {s: backtest(sig, close) for s, sig in sigs.items()}
    )

    # Combined performance + trade statistics table
    for strat_name, sig in sigs.items():
        perf = performance_report(sig, close)
        trades = trade_table(sig, close)
        summary_rows[(stock, strat_name)] = {**perf, **trades}

net_results = pd.concat(results, axis=1)

summary_df = pd.DataFrame.from_dict(summary_rows, orient='index')
summary_df.index.names = ['Stock', 'Strategy']

display(summary_df)

Cumulative Return  Annual Return  Annual Volatility  \
Stock      Strategy                                                          
HDFCBANK   MA                  -0.161545      -0.025676           0.169777   
           Breakout            -0.027305      -0.004079           0.169578   
           RSI                  0.450385       0.056427           0.109425   
           Buy & Hold           0.372891       0.047898           0.255819   
TCS        MA                  -0.013890      -0.002063           0.164361   
           Breakout             0.178867       0.024590           0.169802   
           RSI                  0.064787       0.009310           0.129428   
           Buy & Hold           0.350590       0.045367           0.249450   
RELIANCE   MA                   0.108708       0.015351           0.167696   
           Breakout             0.321793       0.042046           0.173938   
           RSI                 -0.256296      -0.042773           0.155640   
           Buy & Hold           1.009770       0.108543           0.279218   
SUNPHARMA  MA                   1.086655       0.114704           0.180171   
           Breakout             1.108818       0.116444           0.172142   
           RSI                  0.162755       0.022511           0.132051   
           Buy & Hold           3.592450       0.252379           0.242346   
HINDUNILVR MA                  -0.483964      -0.093049           0.160264   
           Breakout            -0.132578      -0.020778           0.169362   
           RSI                 -0.217928      -0.035638           0.118594   
           Buy & Hold           0.176607       0.024300           0.232232   
BHARTIARTL MA                   1.672529       0.156180           0.196068   
           Breakout             2.297907       0.192631           0.201674   
           RSI                  0.022190       0.003245           0.118744   
           Buy & Hold           3.973934       0.267220           0.266276   
M&M        MA                   1.899601       0.170183           0.230713   
           Breakout             2.909849       0.222979           0.240942   
           RSI                 -0.011738      -0.001742           0.170199   
           Buy & Hold           5.913536       0.330340           0.326244   
LT         MA                   0.701275       0.081605           0.185936   
           Breakout             0.868890       0.096714           0.189234   
           RSI                 -0.115507      -0.017957           0.153000   
           Buy & Hold           2.472506       0.201749           0.275802   

                         Sharpe   Sortino  Max Drawdown Max DD Date  \
Stock      Strategy                                                   
HDFCBANK   MA         -0.534087 -0.738066     -0.498471  2024-05-09   
           Breakout   -0.407355 -0.570841     -0.339445  2022-07-18   
           RSI        -0.078348 -0.122751     -0.196908  2026-03-30   
           Buy & Hold -0.066853 -0.094019     -0.404653  2020-03-24   
TCS        MA         -0.408022 -0.594388     -0.451565  2026-11-19   
           Breakout   -0.237982 -0.349730     -0.350105  2026-07-22   
           RSI        -0.430274 -0.609432     -0.224345  2020-03-19   
           Buy & Hold -0.078705 -0.112739     -0.533934  2026-07-01   
RELIANCE   MA         -0.296066 -0.439296     -0.448961  2022-12-23   
           Breakout   -0.131964 -0.197404     -0.416214  2026-09-01   
           RSI        -0.692448 -0.959203     -0.393559  2024-11-21   
           Buy & Hold  0.155948  0.232159     -0.440829  2020-03-23   
SUNPHARMA  MA          0.275873  0.415888     -0.255618  2026-06-09   
           Breakout    0.298848  0.458554     -0.175535  2020-11-20   
           RSI        -0.321763 -0.501170     -0.241312  2020-03-23   
           Buy & Hold  0.773191  1.187461     -0.281032  2020-03-23   
HINDUNILVR MA         -0.986180 -1.355350     -0.504594  2026-07-28   
           Breakout  

In [12]:
summary_df.groupby(level='Strategy').mean(numeric_only=True)

,Cumulative Return,Annual Return,Annual Volatility,Sharpe,Sortino,Max Drawdown,Avg Drawdown,Recovery Days,Total Trades,Win Rate,Avg Win %,Avg Loss %,Largest Win %,Largest Loss %,Avg Holding Days
Strategy,,,,,,,,,,,,,,,
Breakout,0.940780,0.083819,0.185834,0.058899,0.109007,-0.316722,-0.128835,51.500000,24.750,0.445222,0.054596,0.026813,0.410935,0.095924,57.798414
Buy & Hold,2.232786,0.159725,0.265923,0.334613,0.506666,-0.423256,-0.100376,91.333333,0.000,NaN,NaN,NaN,NaN,NaN,NaN
MA,0.601171,0.052154,0.181873,-0.117279,-0.144628,-0.389566,-0.188818,239.000000,20.125,0.409758,0.056327,0.030146,0.424984,0.117654,71.959197
RSI,0.012331,-0.000827,0.135885,-0.478229,-0.679848,-0.307164,-0.120892,919.000000,27.000,0.640017,0.015663,0.014931,0.075451,0.148821,15.983536


In [13]:
summary_df.xs('MA', level='Strategy')['Recovery Days']


,Recovery Days
Stock,
HDFCBANK,NaN
TCS,NaN
RELIANCE,NaN
SUNPHARMA,NaN
HINDUNILVR,NaN
BHARTIARTL,386.0
M&M,92.0
LT,NaN


In [14]:
sig, close = signals['HDFCBANK']['MA'], df1['Close']
pos = sig.shift(1).fillna(0)
ret = pos * close.pct_change().fillna(0) - 0.0025 * pos.diff().abs().fillna(0)
cum = (1 + ret).cumprod()

dd = cum / cum.cummax() - 1
t = dd.idxmin()
peak = cum.cummax().loc[t]

print("trough date:      ", t)
print("peak before it:   ", peak)
print("best value after: ", cum.loc[t:].max())
print("final value:      ", cum.iloc[-1])
print("last date in data:", cum.index[-1])

trough date:       2024-05-09 00:00:00
peak before it:    1.5223082012866347
best value after:  1.012416776102781
final value:       0.8384546180844994
last date in data: 2026-11-23 00:00:00


In [15]:
fee = 0.0025
strat_ret = {}

for stock, df in stocks.items():
    close = df['Close']
    daily_close_ret = close.pct_change()

    for strat_name, sig in signals[stock].items():
        net_ret = (
            sig.shift(1) * daily_close_ret
            - fee * sig.diff().shift(1).abs().fillna(0)
        ).fillna(0)

        strat_ret[(stock, strat_name)] = net_ret

# Optional: combine into a single MultiIndex DataFrame (Dates x (Stock, Strategy))
strat_ret_df = pd.DataFrame(strat_ret)

In [16]:
# 1. Build strat_ret[(stock, strategy)] if not already in memory
fee = 0.0025
strat_ret = {}

for stock, df in stocks.items():
    daily_close_ret = df['Close'].pct_change()
    for strat_name, sig in signals[stock].items():
        strat_ret[(stock, strat_name)] = (
            sig.shift(1) * daily_close_ret
            - fee * sig.diff().shift(1).abs().fillna(0)
        ).fillna(0)

# 2. Classify Nifty 50 (df_b) Trend Regimes: Close vs. 200-day SMA
nifty_close = df_b['Close']
sma_200 = nifty_close.rolling(200).mean()

trend_regime = pd.Series(index=df_b.index, dtype='object')
valid_trend = sma_200.shift(1).notna()
trend_regime[valid_trend] = np.where(
    nifty_close.shift(1)[valid_trend] > sma_200.shift(1)[valid_trend],
    'Bull',
    'Bear'
)

# 3. Classify Nifty 50 Volatility Regimes: 20-day rolling vol vs. median
nifty_ret = nifty_close.pct_change()
roll_vol = nifty_ret.rolling(20).std() * np.sqrt(252)
vol_median = roll_vol.median()

vol_regime = pd.Series(index=df_b.index, dtype='object')
valid_vol = roll_vol.shift(1).notna()
vol_regime[valid_vol] = np.where(
    roll_vol.shift(1)[valid_vol] > vol_median,
    'High Vol',
    'Low Vol'
)

print("Trend regime day counts:\n", trend_regime.value_counts(), "\n")
print("Volatility regime day counts:\n", vol_regime.value_counts())

Trend regime day counts:
 Bull    1145
Bear     360
Name: count, dtype: int64 

Volatility regime day counts:
 Low Vol     843
High Vol    841
Name: count, dtype: int64


In [17]:
rf = 0.065  # 6.5% annual risk-free rate (matching performance_report)

rets = pd.DataFrame(strat_ret)  # columns: (Stock, Strategy)
rets.columns.names = ['Stock', 'Strategy']
regimes = pd.DataFrame({'Trend': trend_regime, 'Vol': vol_regime}).reindex(rets.index)


def regime_metrics(rets_df, regime_series, rf=0.065):
    # Share of valid (post-warmup) days spent in each regime
    day_share = regime_series.value_counts(normalize=True).round(4)

    # Annualized arithmetic mean return, volatility, and excess Sharpe per (Stock, Strategy)
    ann_ret = rets_df.groupby(regime_series).mean() * 252
    ann_vol = rets_df.groupby(regime_series).std() * np.sqrt(252)
    sharpe = (ann_ret - rf) / ann_vol.replace(0, np.nan)

    # Average across all 8 stocks by Strategy (rows: Strategy, cols: Regime)
    by_strat_ret = ann_ret.T.groupby(level='Strategy', sort=False).mean()
    by_strat_vol = ann_vol.T.groupby(level='Strategy', sort=False).mean()
    by_strat_sharpe = sharpe.T.groupby(level='Strategy', sort=False).mean()

    return {
        "day_share": day_share,
        "ann_ret": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "strat_ret": by_strat_ret,
        "strat_vol": by_strat_vol,
        "strat_sharpe": by_strat_sharpe,
    }


trend_res = regime_metrics(rets, regimes['Trend'], rf=rf)
vol_res = regime_metrics(rets, regimes['Vol'], rf=rf)

print("Trend Regime Day Share:\n", trend_res["day_share"], "\n")
print("Volatility Regime Day Share:\n", vol_res["day_share"])

Trend Regime Day Share:
 Trend
Bull    0.7608
Bear    0.2392
Name: proportion, dtype: float64 

Volatility Regime Day Share:
 Vol
Low Vol     0.5006
High Vol    0.4994
Name: proportion, dtype: float64


In [18]:
sector_map = {
    "HDFCBANK": "Financials",
    "TCS": "IT",
    "RELIANCE": "Energy",
    "SUNPHARMA": "Healthcare",
    "HINDUNILVR": "FMCG",
    "BHARTIARTL": "Telecom",
    "M&M": "Auto",
    "LT": "Industrials",
}


def group_by_sector(metric_df, sector_map):
    df_t = metric_df.T.copy()
    df_t['Sector'] = df_t.index.get_level_values('Stock').map(sector_map)
    return df_t.groupby(['Sector', 'Strategy'], sort=False).mean().round(3)


trend_sector_ret = group_by_sector(trend_res["ann_ret"], sector_map)
trend_sector_sharpe = group_by_sector(trend_res["sharpe"], sector_map)

vol_sector_ret = group_by_sector(vol_res["ann_ret"], sector_map)
vol_sector_sharpe = group_by_sector(vol_res["sharpe"], sector_map)

print("=== Trend Regime: Annualized Return by Sector & Strategy ===")
display(trend_sector_ret)

print("=== Volatility Regime: Sharpe (rf=6.5%) by Sector & Strategy ===")
display(vol_sector_sharpe)

=== Trend Regime: Annualized Return by Sector & Strategy ===


Trend                    Bear   Bull
Sector      Strategy                
Financials  MA          0.087 -0.076
            Breakout    0.060  0.013
            RSI         0.067  0.028
            Buy & Hold  0.264  0.030
IT          MA         -0.062 -0.043
            Breakout   -0.102  0.035
            RSI         0.110  0.016
            Buy & Hold  0.010  0.042
Energy      MA         -0.054 -0.035
            Breakout   -0.228  0.029
            RSI         0.115 -0.053
            Buy & Hold  0.251  0.027
Healthcare  MA          0.092  0.148
            Breakout    0.155  0.135
            RSI         0.088  0.004
            Buy & Hold  0.261  0.255
FMCG        MA         -0.026 -0.059
            Breakout    0.097 -0.036
            RSI        -0.031 -0.051
            Buy & Hold  0.176 -0.020
Telecom     MA          0.079  0.212
            Breakout    0.125  0.238
            RSI         0.014  0.039
            Buy & Hold  0.327  0.311
Auto        MA          0.174  0.117
            Breakout    0.192  0.210
            RSI         0.018  0.071
            Buy & Hold  0.264  0.373
Industrials MA         -0.077  0.192
            Breakout   -0.096  0.214
            RSI         0.082  0.025
            Buy & Hold  0.205  0.326

=== Volatility Regime: Sharpe (rf=6.5%) by Sector & Strategy ===


Vol                     High Vol  Low Vol
Sector      Strategy                     
Financials  MA            -0.062   -0.961
            Breakout      -0.215   -0.482
            RSI            0.158   -0.512
            Buy & Hold     0.218   -0.167
IT          MA            -0.081   -0.589
            Breakout      -0.122   -0.183
            RSI           -0.160   -0.830
            Buy & Hold     0.244   -0.225
Energy      MA             0.002   -0.526
            Breakout       0.118   -0.274
            RSI           -0.736   -0.509
            Buy & Hold     0.530   -0.103
Healthcare  MA             0.434    0.222
            Breakout       0.632    0.007
            RSI           -0.196   -0.473
            Buy & Hold     1.158    0.211
FMCG        MA            -1.088   -0.758
            Breakout      -0.366   -0.510
            RSI           -0.485   -1.264
            Buy & Hold    -0.033   -0.202
Telecom     MA             0.128    0.977
            Breakout       0.269    1.157
            RSI           -0.410   -0.983
            Buy & Hold     0.358    1.366
Auto        MA             0.874    0.140
            Breakout       0.950    0.387
            RSI           -0.226   -0.621
            Buy & Hold     1.017    0.551
Industrials MA            -0.251    0.614
            Breakout       0.232    0.249
            RSI           -0.622   -0.212
            Buy & Hold     0.191    1.109

In [19]:
gold_df = data('GOLDBEES.NS', '2020-01-01', date.today())
gold_close = gold_df['Close'].reindex(rets.index).ffill()
gold_sig = pd.Series(1, index=rets.index)

gold_ret = (
    gold_sig.shift(1) * gold_close.pct_change()
    - fee * gold_sig.diff().shift(1).abs().fillna(0)
).fillna(0)


def build_comparison_table(res_dict, regime_series, gold_ret, rf=0.065):
    gold_ann_ret = gold_ret.groupby(regime_series).mean() * 252
    gold_ann_vol = gold_ret.groupby(regime_series).std() * np.sqrt(252)
    gold_sharpe = (gold_ann_ret - rf) / gold_ann_vol.replace(0, np.nan)

    ret_table = res_dict["strat_ret"].copy()
    ret_table.loc["GOLDBEES (Buy & Hold)"] = gold_ann_ret

    sharpe_table = res_dict["strat_sharpe"].copy()
    sharpe_table.loc["GOLDBEES (Buy & Hold)"] = gold_sharpe

    combined = pd.concat(
        {"Annual Return": ret_table, "Sharpe (rf=6.5%)": sharpe_table},
        axis=1
    ).round(3)
    return combined


print("=== Trend Regime Comparison (Equities vs. GOLDBEES.NS) ===")
display(build_comparison_table(trend_res, regimes['Trend'], gold_ret, rf=rf))

print("=== Volatility Regime Comparison (Equities vs. GOLDBEES.NS) ===")
display(build_comparison_table(vol_res, regimes['Vol'], gold_ret, rf=rf))

[*********************100%***********************]  1 of 1 completed

=== Trend Regime Comparison (Equities vs. GOLDBEES.NS) ===


Annual Return        Sharpe (rf=6.5%)       
Trend                          Bear   Bull             Bear   Bull
Strategy                                                          
MA                            0.027  0.057           -0.298 -0.105
Breakout                      0.025  0.105           -0.335  0.152
RSI                           0.058  0.010           -0.093 -0.736
Buy & Hold                    0.220  0.168            0.601  0.415
GOLDBEES (Buy & Hold)         0.239  0.128            0.814  0.387

=== Volatility Regime Comparison (Equities vs. GOLDBEES.NS) ===


Annual Return         Sharpe (rf=6.5%)        
Vol                        High Vol Low Vol         High Vol Low Vol
Strategy                                                            
MA                            0.072   0.058           -0.005  -0.110
Breakout                      0.111   0.080            0.187   0.044
RSI                           0.003   0.013           -0.334  -0.675
Buy & Hold                    0.216   0.136            0.460   0.317
GOLDBEES (Buy & Hold)         0.105   0.244            0.217   0.990